# Part 2 - Fine-Tuning YOLO26 to Read Sign Language (ASL A-Z)

In **Part 1** we taught YOLO26 to spot safety gear on a construction site. Now we apply the *exact same fine-tuning recipe* to a completely different problem: **real-time sign-language recognition**. We'll fine-tune YOLO26 to recognise the **American Sign Language (ASL) alphabet (A-Z)** from a single hand, then read letters live from a video and spell them into words.

## What you'll learn
- How the same fine-tuning workflow transfers to a brand-new domain
- How to load a 26-class ASL dataset (already in YOLO format)
- How to fine-tune **YOLO26**, evaluate it, and run it on images + video
- How to turn per-frame letter detections into a live **spelled word**

## Why fine-tune?
The pretrained YOLO26 (trained on COCO) knows 80 everyday objects - but it has **zero concept of a hand-sign letter**. Sign language is how millions of people communicate, and a tool that reads fingerspelling from a camera is a genuine accessibility aid. To build it, the model must learn **26 brand-new classes it has never seen** - which is exactly what fine-tuning does.


## 1. Setup

In [ ]:
%pip install -q ultralytics roboflow gdown
import ultralytics
ultralytics.checks()

## 2. Configuration
Everything you might tweak lives here. The asset link is the only thing you'll change later - swap the Google Drive folder for a GitHub Release URL and nothing else moves.

In [ ]:
import os, glob

MODEL_WEIGHTS = "yolo26n.pt"
WORKSPACE = "david-lee-d0rhs"
PROJECT   = "american-sign-language-letters"
VERSION   = 6
FORMAT    = "yolov8"
EPOCHS    = 100
IMGSZ     = 416
RUN_NAME  = "asl_finetune"

ASSETS_DIR = "assets"
ASSETS_URL = "https://github.com/spmallick/learnopencv/releases/download/v1.0/Assets.zip"

print("Dataset:", PROJECT, "v" + str(VERSION), "| Model:", MODEL_WEIGHTS)

## 3. Roboflow API key
The dataset lives on Roboflow, which needs a **free** API key (roboflow.com -> Settings -> API Keys). Paste yours below - the same single step whether you run on Colab, Kaggle, or your own machine.

In [ ]:
# Get a FREE key at roboflow.com -> Settings -> API Keys, then paste it here:
API_KEY = "0c4XQBsECsYHA7WKoJnW"

assert API_KEY != "PASTE_YOUR_ROBOFLOW_KEY", "Paste your Roboflow API key above."

## 4. Download the dataset
Pulls the **ASL Letters** dataset (Roboflow, CC0 / public domain) already in YOLO format - images, labels and a ready `data.yaml`.

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=API_KEY)
ds = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download(FORMAT)
DATA_YAML = os.path.join(ds.location, "data.yaml")
print("Dataset ready at:", ds.location)

## 5. Download the test assets

In [ ]:
import os, glob, urllib.request, zipfile

# Download the test assets from the GitHub Release (once)
if not glob.glob(os.path.join(ASSETS_DIR, "**", "*"), recursive=True):
    urllib.request.urlretrieve(ASSETS_URL, "Assets.zip")
    zipfile.ZipFile("Assets.zip").extractall(ASSETS_DIR)

def asset(name):
    hits = glob.glob(os.path.join(ASSETS_DIR, "**", name), recursive=True)
    if not hits:
        raise FileNotFoundError(f"'{name}' not found under {ASSETS_DIR}/ - check ASSETS_URL.")
    return hits[0]

print("Assets folder:", os.path.abspath(ASSETS_DIR))

## 6. Explore the dataset
The 26 letter classes, image counts per split, class balance, and a few labelled samples.

In [ ]:
import yaml
from pathlib import Path

cfg = yaml.safe_load(open(DATA_YAML))
_names = cfg["names"]
NAMES = {int(k): v for k, v in _names.items()} if isinstance(_names, dict) else {i: n for i, n in enumerate(_names)}
print(f"Classes ({len(NAMES)}):", NAMES)

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
def list_images(d):
    p = Path(d)
    return [q for q in p.rglob("*") if q.suffix.lower() in IMG_EXT] if p.exists() else []
def split_dir(split):
    for c in [Path(ds.location)/split/"images", Path(ds.location)/split]:
        if c.exists(): return c
    return Path(ds.location)/split/"images"

TRAIN_IMAGES = list_images(split_dir("train"))
VAL_IMAGES   = list_images(split_dir("valid")) or list_images(split_dir("val")) or list_images(split_dir("test"))
print("Train images:", len(TRAIN_IMAGES), "| Val images:", len(VAL_IMAGES))

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

def yolo_label_path(p):
    parts = list(Path(p).parts)
    for i in range(len(parts) - 1, -1, -1):
        if parts[i] == "images":
            parts[i] = "labels"; break
    return Path(*parts).with_suffix(".txt")

counts = Counter()
for img in TRAIN_IMAGES:
    lp = yolo_label_path(img)
    if lp.exists():
        for line in lp.read_text().splitlines():
            if line.strip():
                counts[int(float(line.split()[0]))] += 1
if counts:
    keys = sorted(counts)
    labels = [NAMES.get(c, str(c)) for c in keys]
    vals   = [counts[c] for c in keys]
    plt.figure(figsize=(12, 4)); plt.bar(labels, vals, color="#4C9AFF")
    plt.title("Training boxes per class (A-Z)"); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

In [ ]:
import cv2, random

def find_label(p):
    p = Path(p)
    c = yolo_label_path(p)                                  # standard images/ -> labels/
    if c.exists(): return c
    if p.with_suffix(".txt").exists(): return p.with_suffix(".txt")   # same-folder fallback
    hits = list(Path(ds.location).rglob(p.stem + ".txt"))  # last resort: search the dataset
    return hits[0] if hits else None

def draw_boxes(p):
    im = cv2.imread(str(p))
    if im is None: return None
    h, w = im.shape[:2]
    lp = find_label(p)
    if lp is None:
        cv2.putText(im, "no label found", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
        return im[:, :, ::-1]
    scale = max(0.7, w / 800)                               # label size scales with image
    th = max(2, int(w / 500))
    for line in lp.read_text().splitlines():
        if not line.strip(): continue
        c, xc, yc, bw, bh = map(float, line.split()[:5])
        x1, y1 = int((xc - bw/2)*w), int((yc - bh/2)*h)
        x2, y2 = int((xc + bw/2)*w), int((yc + bh/2)*h)
        cv2.rectangle(im, (x1, y1), (x2, y2), (0, 200, 0), th)
        label = str(NAMES.get(int(c), int(c)))
        (tw, tht), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, scale, th)
        cv2.rectangle(im, (x1, y1 - tht - 10), (x1 + tw + 8, y1), (0, 200, 0), -1)   # filled bg
        cv2.putText(im, label, (x1 + 4, y1 - 6), cv2.FONT_HERSHEY_SIMPLEX, scale, (255, 255, 255), th, cv2.LINE_AA)
    return im[:, :, ::-1]

samples = random.sample(TRAIN_IMAGES, min(6, len(TRAIN_IMAGES)))
plt.figure(figsize=(14, 8))
for i, p in enumerate(samples):
    rgb = draw_boxes(p)
    if rgb is None: continue
    plt.subplot(2, 3, i + 1); plt.imshow(rgb); plt.axis("off")
plt.suptitle("Sample ASL training images with ground-truth letter boxes"); plt.tight_layout(); plt.show()

## 7. First, watch the base model fail (this is the "why")
COCO has no hand-sign letters at all, so the stock YOLO26 is completely blind to ASL. We run it on a clear hand image - it cannot output any letter. Confirming this is the whole motivation for fine-tuning.

In [ ]:
from ultralytics import YOLO

base = YOLO(MODEL_WEIGHTS)
print(f"All {len(NAMES)} ASL letter classes are absent from COCO - the base model can detect none of them.")

img = asset("yolo26_Finetuning_image_ASL_B.jpg")
r = base.predict(img, conf=0.25, verbose=False)[0]
detected = [base.names[int(c)] for c in r.boxes.cls.tolist()] if (r.boxes is not None and len(r.boxes)) else []
print("Base model detected:", detected or "nothing useful (no letter)")
plt.figure(figsize=(7, 7)); plt.imshow(r.plot()[:, :, ::-1]); plt.axis("off")
plt.title("Pretrained YOLO26 (COCO): no idea this is the letter 'B'"); plt.show()

## 8. Fine-tune YOLO26
We start from the pretrained `yolo26n.pt` and teach it the 26 letters. On a GPU this takes only a few minutes. Outputs land under `runs_yolo26/`.

In [ ]:
model = YOLO(MODEL_WEIGHTS)
results = model.train(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=16, patience=20,
    project="runs_yolo26", name=RUN_NAME, exist_ok=True,
)
SAVE_DIR = Path(model.trainer.save_dir)
print("Training complete ->", SAVE_DIR)

## 9. Evaluate
mAP scores plus the saved plots. The confusion matrix is especially interesting here - look for the look-alike clusters (U/V/F, and the fist letters) bleeding into each other.

In [ ]:
metrics = model.val()
print(f"mAP50-95: {metrics.box.map:.4f} | mAP50: {metrics.box.map50:.4f}")

from IPython.display import Image, display
for pattern in ["results.png", "confusion_matrix.png", "*PR_curve.png"]:
    for f in sorted(SAVE_DIR.glob(pattern)):
        print(f.name); display(Image(filename=str(f)))

## 10. Test it on images
Run the fine-tuned model on our four test letter images (pulled by name from `assets/`). The same `A` image that the base model couldn't read should now be labelled correctly.

In [ ]:
import cv2, numpy as np

best = YOLO(SAVE_DIR / "weights" / "best.pt")
names = [
    "yolo26_Finetuning_image_ASL_O.jpg",
    "yolo26_Finetuning_image_ASL_B.jpg",
    "yolo26_Finetuning_image_ASL_H.jpg",
    "yolo26_Finetuning_image_ASL_U.jpg",
]

def to_square(path, size=640, pad=235):
    img = cv2.imread(str(path))
    h, w = img.shape[:2]
    s = size / max(h, w)
    rs = cv2.resize(img, (int(w * s), int(h * s)))
    canvas = np.full((size, size, 3), pad, np.uint8)
    y, x = (size - rs.shape[0]) // 2, (size - rs.shape[1]) // 2
    canvas[y:y + rs.shape[0], x:x + rs.shape[1]] = rs
    return canvas

squares = [to_square(asset(n)) for n in names]
preds = best.predict(squares, conf=0.20, verbose=False)

plt.figure(figsize=(10, 10))
for i, r in enumerate(preds):
    ann = r.plot(conf=False, line_width=4, font_size=24)   # conf=False -> label shows just the letter
    plt.subplot(2, 2, i + 1); plt.imshow(ann[:, :, ::-1]); plt.axis("off")
plt.suptitle("Fine-tuned YOLO26 - ASL letter detection"); plt.tight_layout(); plt.show()

## 11. Real-time video - spell a word
Run the model on the gesture video. We read the top-confidence letter each frame and, once a letter is held steady for a few frames, append it to a growing **word** (a simple debounce so transition frames don't spam letters).

In [ ]:
VIDEO = asset("yolo26_Finetuning_video_ASL_gestures.mp4")
best = YOLO(SAVE_DIR / "weights" / "best.pt")
NAMES = best.names

OUT = "asl_annotated.mp4"
cap = cv2.VideoCapture(VIDEO); fps = cap.get(cv2.CAP_PROP_FPS) or 25
W, writer = 1280, None

while True:
    ok, frame = cap.read()
    if not ok: break
    frame = cv2.resize(frame, (W, int(W * frame.shape[0] / frame.shape[1])))
    r = best.predict(frame, conf=0.1, verbose=False)[0]
    ann = r.plot()
    letter = None
    if r.boxes is not None and len(r.boxes):
        top = int(r.boxes.conf.argmax()); letter = NAMES[int(r.boxes.cls[top])]
    cv2.putText(ann, f"Letter: {letter or '-'}", (20, 48), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (0, 255, 255), 2, cv2.LINE_AA)
    if writer is None:
        h2, w2 = ann.shape[:2]
        writer = cv2.VideoWriter(OUT, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w2, h2))
    writer.write(ann)

cap.release()
if writer: writer.release()
print("Saved:", OUT)

In [ ]:
import shutil
play = OUT
if shutil.which("ffmpeg"):
    import subprocess
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", OUT,
                    "-vcodec", "libx264", "-pix_fmt", "yuv420p", "asl_play.mp4"])
    play = "asl_play.mp4"

from IPython.display import HTML
from base64 import b64encode
HTML('<video width=720 controls><source src="data:video/mp4;base64,' +
     b64encode(open(play, "rb").read()).decode() + '" type="video/mp4"></video>')

## 12. Export for deployment

In [ ]:
best.export(format="onnx")   # also: "engine", "coreml", "tflite", "openvino"

## Recap & series wrap-up

Same recipe as Part 1, you fine-tuned **YOLO26** to read the **ASL alphabet**, then spelled letters live from video.

**Honest limitations:** ASL letters include look-alike clusters (U/V/F, and the fist letters), and the dataset is small and uniform - so a hand at an odd angle or on a busy background can misfire. Hold each letter ~1s on a plain background, and raise `conf`/`HOLD` to clean up flickery reads.

Across **Part 1 (Construction PPE)** and **Part 2 (ASL)** you've now applied the *same* fine-tuning workflow to two very different real-world problems - which is the whole point: **the recipe transfers.**

And that is all for now. Pick a dataset you care about, fine-tune YOLO26 on it, and see what you can teach it to spot.